# CV03 — XGBoost Model (Machine Learning)
**Thành viên:** Trần Thị Thanh Phương  
**Lab:** 4

Notebook thực nghiệm sử dụng các hàm trong `src/cv03_xgboost_model.py`. Chạy các ô theo thứ tự từ trên xuống.

## 0. Chuẩn bị môi trường và dữ liệu

In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import numpy as np
import joblib

REPO = Path('/content/QLCTG_nhom7')
LAB4 = REPO / 'time-series-lab4'
if not (LAB4 / 'src' / 'cv03_xgboost_model.py').exists():
    if REPO.exists():
        raise FileNotFoundError('Đã có thư mục repository nhưng thiếu module CV03. Kiểm tra lại thư mục trước khi tiếp tục.')
    subprocess.run(['git', 'clone', '--branch', 'phuong-lab04',
                    'https://github.com/HADUYHUNG-0912/QLCTG_nhom7.git', str(REPO)], check=True)

if not (LAB4 / 'src' / 'cv03_xgboost_model.py').exists():
    raise FileNotFoundError('Không tìm thấy src/cv03_xgboost_model.py')

# Tương thích Google Colab: chỉ gọi reconfigure nếu stdout hỗ trợ.
module_path = LAB4 / 'src' / 'cv03_xgboost_model.py'
module_source = module_path.read_text(encoding='utf-8')
old_line = 'sys.stdout.reconfigure(encoding="utf-8")'
new_lines = 'if hasattr(sys.stdout, "reconfigure"):\n    sys.stdout.reconfigure(encoding="utf-8")'
if old_line in module_source and 'if hasattr(sys.stdout, "reconfigure"):' not in module_source:
    module_path.write_text(module_source.replace(old_line, new_lines), encoding='utf-8')

sys.path.insert(0, str(LAB4 / 'src'))
from cv03_xgboost_model import (
    step1_build_flat_features, step2_train_xgboost,
    step3_evaluate, step4_feature_importance, step5_answer_questions,
    LAG, PROCESSED_DIR
)
print('Đã sẵn sàng:', LAB4)
print('Lag:', LAG)

Đã sẵn sàng: /content/QLCTG_nhom7/time-series-lab4
Lag: 24


## Step 1: Build Flat Feature Matrix (96 lags + rolling)

Tạo các biến trễ trên dữ liệu theo đúng thứ tự thời gian. Các dòng Train thiếu lịch sử 24 bước đầu được loại bỏ; Validation và Test sử dụng lịch sử trước đó để tạo đặc trưng.

In [2]:
train = pd.read_csv(PROCESSED_DIR / 'train.csv', index_col=0, parse_dates=True)
val = pd.read_csv(PROCESSED_DIR / 'val.csv', index_col=0, parse_dates=True)
test = pd.read_csv(PROCESSED_DIR / 'test.csv', index_col=0, parse_dates=True)

combined = pd.concat([train, val, test])
features = step1_build_flat_features(combined, lag=LAG)
train_features = features.loc[train.index].dropna()
val_features = features.loc[val.index].dropna()
test_features = features.loc[test.index].dropna()

TARGET_COL = 'load_t0'
assert TARGET_COL in features.columns, f'Thiếu cột mục tiêu: {TARGET_COL}'
feature_cols = [c for c in features.columns if c != TARGET_COL]
X_train, y_train = train_features[feature_cols], train_features[TARGET_COL]
X_val, y_val = val_features[feature_cols], val_features[TARGET_COL]
X_test, y_test = test_features[feature_cols], test_features[TARGET_COL]
print('X_train:', X_train.shape, '| X_val:', X_val.shape, '| X_test:', X_test.shape)
print('Số đặc trưng:', len(feature_cols))
assert len(feature_cols) == 99, 'Kiểm tra lại số đặc trưng theo module CV03'

[STEP 1] Xây dựng flat features cho XGBoost...
  Số features tạo thành : 99 (96 lags + 3 rolling/trend)
X_train: (18387, 99) | X_val: (3945, 99) | X_test: (3947, 99)
Số đặc trưng: 99


## Step 2: Train XGBoost with Early Stopping

Huấn luyện theo cấu hình trong module CV03; sử dụng tập Validation để theo dõi quá trình học.

In [3]:
model, evals_result = step2_train_xgboost(X_train, y_train, X_val, y_val)
print('Best iteration:', getattr(model, 'best_iteration', 'Không có'))
print('Nhóm tập đánh giá:', list(evals_result.keys()))

[STEP 2] Huấn luyện XGBoost Regressor
[0]	validation_0-rmse:0.98280	validation_1-rmse:1.05842
[100]	validation_0-rmse:0.24743	validation_1-rmse:0.27028
[200]	validation_0-rmse:0.18593	validation_1-rmse:0.21418
[300]	validation_0-rmse:0.17207	validation_1-rmse:0.20886
[400]	validation_0-rmse:0.16346	validation_1-rmse:0.20658
[500]	validation_0-rmse:0.15653	validation_1-rmse:0.20507
[600]	validation_0-rmse:0.15088	validation_1-rmse:0.20408
[700]	validation_0-rmse:0.14589	validation_1-rmse:0.20336
[800]	validation_0-rmse:0.14108	validation_1-rmse:0.20262
[900]	validation_0-rmse:0.13705	validation_1-rmse:0.20197
[999]	validation_0-rmse:0.13310	validation_1-rmse:0.20162

  Số cây tối ưu (Best iteration) : 996
  Best Val RMSE (scaled)        : 0.20160
  Đã lưu biểu đồ Learning Curve: /content/QLCTG_nhom7/time-series-lab4/outputs/figures/cv03_xgb_learning_curve.png
[STEP 2 DONE]

Best iteration: 996
Nhóm tập đánh giá: ['validation_0', 'validation_1']


## Step 3: Evaluate & Feature Importance

Đánh giá trên tập Test và xem mức độ quan trọng của các đặc trưng.

In [4]:
scaler = joblib.load(PROCESSED_DIR / 'scaler.pkl')
metrics, y_true, y_pred = step3_evaluate(model, X_test, y_test, scaler)
print('=== CHỈ SỐ ĐÁNH GIÁ TEST ===')
for key, value in metrics.items():
    print(f'{key}: {value}')

fi_df = step4_feature_importance(model, feature_cols)
print('\n=== TOP FEATURE IMPORTANCE ===')
display(fi_df.head(15))

[STEP 3] Đánh giá XGBoost trên tập Test
  RMSE = 478.3519 MW
  MAE  = 331.0779 MW
  MAPE = 2.1776 %
  Đã lưu đồ thị: /content/QLCTG_nhom7/time-series-lab4/outputs/figures/cv03_xgb_forecast_vs_actual.png
[STEP 3 DONE]

=== CHỈ SỐ ĐÁNH GIÁ TEST ===
Model: XGBoost
RMSE: 478.3519
MAE: 331.0779
MAPE: 2.1776
[STEP 4] Phân tích Feature Importance (XGBoost Gain)

Top 15 đặc trưng có Gain cao nhất:
        feature  importance  rank
      load_lag1  170.714188     1
     load_lag24  154.259598     2
     load_lag23   37.910980     3
     load_lag22   18.447550     4
      load_lag2   16.095842     5
     load_lag21   10.059232     6
      temp_lag1    7.933810     7
     load_trend    7.413203     8
      temp_lag2    6.813896     9
     load_lag20    4.067292    10
rolling_mean_6h    3.278612    11
 humidity_lag19    2.083528    12
 humidity_lag18    2.031544    13
      load_lag6    2.020687    14
 humidity_lag20    1.983215    15

  Đã lưu: /content/QLCTG_nhom7/time-series-lab4/outputs/tables

,feature,importance,rank
0,load_lag1,170.714188,1
1,load_lag24,154.259598,2
2,load_lag23,37.910980,3
3,load_lag22,18.447550,4
4,load_lag2,16.095842,5
5,load_lag21,10.059232,6
6,temp_lag1,7.933810,7
7,load_trend,7.413203,8
8,temp_lag2,6.813896,9
9,load_lag20,4.067292,10
